In [2]:
!pip install ultralytics opencv-python-headless filterpy lap scipy

  Using cached ultralytics-8.4.52-py3-none-any.whl.metadata (39 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached ultralytics_thop-2.0.19-py3-none-any.whl.metadata (14 kB)
Using cached ultralytics-8.4.52-py3-none-any.whl (1.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.4 MB/s eta 0:00:00
Using cached ultralytics_thop-2.0.19-py3-none-any.whl (28 kB)
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=1a061449515052e51c506cf8b94852a741c8e8f8c6af5fc2eb68db49e133e3e1
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy


In [3]:
!wget -O sort.py https://raw.githubusercontent.com/abewley/sort/master/sort.py

--2026-05-20 13:02:27--  https://raw.githubusercontent.com/abewley/sort/master/sort.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11739 (11K) [text/plain]
Saving to: ‘sort.py’

sort.py             100%[===================>]  11.46K  --.-KB/s    in 0s      

2026-05-20 13:02:27 (105 MB/s) - ‘sort.py’ saved [11739/11739]



In [6]:
# Fix matplotlib backend issue in Colab

with open("sort.py", "r") as file:
    content = file.read()

content = content.replace(
    "matplotlib.use('TkAgg')",
    "matplotlib.use('Agg')"
)

with open("sort.py", "w") as file:
    file.write(content)

print("SORT backend fixed successfully.")

SORT backend fixed successfully.


In [7]:
from google.colab import files

uploaded = files.upload()

Saving traffic.mp4 to traffic (1).mp4


In [13]:
import cv2
import numpy as np
from ultralytics import YOLO
from sort import Sort
from google.colab.patches import cv2_imshow

# -----------------------------
# Configuration
# -----------------------------
VIDEO_PATH = "traffic.mp4"
OUTPUT_PATH = "output.mp4"

CONF_THRESHOLD = 0.4

# COCO vehicle classes
VEHICLE_CLASSES = [2, 3, 5, 7]

DENSITY_THRESHOLD = 5

# Signal timing
FIXED_GREEN = 20
MIN_GREEN = 10
MAX_GREEN = 60

# -----------------------------
# Load YOLO Model
# -----------------------------
model = YOLO("yolov8n.pt")

# -----------------------------
# Initialize SORT
# -----------------------------
tracker = Sort(
    max_age=20,
    min_hits=3,
    iou_threshold=0.3
)

# -----------------------------
# Helper Functions
# -----------------------------
def estimate_density(count):

    if count < 2:
        return "Low"

    elif count < 5:
        return "Medium"

    else:
        return "High"


def adaptive_signal_time(count):

    green_time = MIN_GREEN + (count * 2)

    return min(MAX_GREEN, green_time)

# -----------------------------
# Open Video
# -----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps = int(cap.get(cv2.CAP_PROP_FPS))

if fps == 0:
    fps = 20

# -----------------------------
# Output Writer
# -----------------------------
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (frame_width, frame_height)
)

tracked_vehicle_ids = set()

frame_count = 0

# -----------------------------
# Main Processing Loop
# -----------------------------
while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    frame_count += 1

    # -----------------------------
    # YOLO Detection
    # -----------------------------
    results = model(frame, verbose=False)[0]

    detections = []

    for box in results.boxes:

        cls = int(box.cls[0])
        conf = float(box.conf[0])

        if cls in VEHICLE_CLASSES and conf > CONF_THRESHOLD:

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            detections.append([
                x1,
                y1,
                x2,
                y2,
                conf
            ])

    detections = np.array(detections)

    # -----------------------------
    # SORT Tracking
    # -----------------------------
    if len(detections) > 0:

        tracked_objects = tracker.update(detections)

    else:

        tracked_objects = np.empty((0, 5))

    current_vehicle_count = 0

    # -----------------------------
    # Draw Tracking Results
    # -----------------------------
    for track in tracked_objects:

        x1, y1, x2, y2, track_id = track

        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

        current_vehicle_count += 1

        tracked_vehicle_ids.add(int(track_id))

        # Bounding box
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        # Vehicle ID
        cv2.putText(
            frame,
            f"ID: {int(track_id)}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 255),
            2
        )

    # -----------------------------
    # Density Estimation
    # -----------------------------
    density = estimate_density(current_vehicle_count)

    # -----------------------------
    # Adaptive Signal Timing
    # -----------------------------
    fixed_time = FIXED_GREEN

    adaptive_time = adaptive_signal_time(current_vehicle_count)

    # -----------------------------
    # Display Information
    # -----------------------------
    cv2.putText(
        frame,
        f"Vehicles: {current_vehicle_count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Density: {density}",
        (20, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Fixed Green: {fixed_time}s",
        (20, 120),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 0),
        2
    )

    cv2.putText(
        frame,
        f"Adaptive Green: {adaptive_time}s",
        (20, 160),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 255),
        2
    )

    # -----------------------------
    # Congestion Alert
    # -----------------------------
    if current_vehicle_count > DENSITY_THRESHOLD:

        cv2.putText(
            frame,
            "CONGESTION ALERT!",
            (20, 220),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            3
        )

    # -----------------------------
    # Save Output Frame
    # -----------------------------
    out.write(frame)

    # -----------------------------
    # Show Sample Frames
    # -----------------------------
    if frame_count % 100 == 0:

        print(f"Processed {frame_count} frames")

# -----------------------------
# Release Resources
# -----------------------------
cap.release()
out.release()

print("\nProcessing Complete")
print(f"Output saved as: {OUTPUT_PATH}")

print(f"\nTotal Unique Vehicles Tracked: {len(tracked_vehicle_ids)}")

print("\nAdaptive Traffic Signal Prototype Executed Successfully")

Processed 100 frames
Processed 200 frames
Processed 300 frames
Processed 400 frames
Processed 500 frames
Processed 600 frames
Processed 700 frames
Processed 800 frames
Processed 900 frames
Processed 1000 frames
Processed 1100 frames

Processing Complete
Output saved as: output.mp4

Total Unique Vehicles Tracked: 396

Adaptive Traffic Signal Prototype Executed Successfully


In [11]:
from google.colab import files

files.download("output.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>